# Self-hosted model vs Groq: the LLM benchmark on vLLM

Serves an open code model with **vLLM** on Kaggle's free T4 GPU, then runs this project's benchmark (`eval/llm_benchmark.py`: the 30 cases, the production prompts, the production scoring) against it. Compare the result with the Groq `gpt-oss-120b` run on your laptop with `--replay`.

**Before running:** Settings (right panel) → Accelerator **GPU T4 x2** (not P100: recent vLLM does not support it), and **Internet on** (needs a phone-verified Kaggle account). Then *Run all*. See `docs/SERVING.md`.

## 1. Settings

The model is the 4-bit (AWQ) Qwen2.5-Coder-7B: it fits one T4 (16 GB) with room for the ~5k-token prompts. `RUNS = 1` first; set 3 for a result you can decide on (about 1-2 hours).

In [ ]:
import os, secrets

REPO_URL = "https://github.com/montahalassoued/Autonomous_Code_Reviewer.git"
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"
SERVED_NAME = "qwen2.5-coder-7b"   # the name the benchmark asks for: vllm:qwen2.5-coder-7b
RUNS = 1
ONLY = ""                          # e.g. "sec_sql_injection,logic_clean" for a quick test; "" = all 30 cases

API_KEY = secrets.token_hex(16)    # the server only answers requests carrying this key
WORK = "/kaggle/working"
TMP = "/kaggle/tmp"
os.makedirs(TMP, exist_ok=True)

## 2. Check the GPU

Expect a **Tesla T4** line. "command not found" means the accelerator is off: turn it on in Settings and restart.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Two separate environments

vLLM and this project pin different versions of the same libraries (torch, openai, pydantic), so each gets its own virtual environment. `uv` installs in seconds what `pip` takes minutes to do. vLLM is the slow one (a few minutes).

In [ ]:
!pip install -q uv
!uv venv -q {TMP}/vllm-env --python 3.11 && uv pip install -q --python {TMP}/vllm-env/bin/python vllm
!git clone -q --depth 1 {REPO_URL} {TMP}/reviewer
!uv venv -q {TMP}/app-env --python 3.11 && uv pip install -q --python {TMP}/app-env/bin/python -r {TMP}/reviewer/requirements.txt
!{TMP}/vllm-env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

## 4. Start the vLLM server in the background

- `--dtype half`: the T4 has no bfloat16, so the model runs in float16.
- `--max-model-len 16384`: the longest prompt + answer the server accepts; it reserves GPU memory for it.
- `--gpu-memory-utilization 0.90`: vLLM takes 90% of the GPU for the weights and the KV cache (its memory of the requests in flight).
- `--enable-auto-tool-choice --tool-call-parser hermes`: lets the model call tools (needed by the review eval; the benchmark sends none). `hermes` is Qwen's tool-call format.
- `--api-key`: requests without the key are refused, even here.

The log goes to `vllm.log`; the next cell waits until the server answers.

In [ ]:
import subprocess

server = subprocess.Popen(
    [f"{TMP}/vllm-env/bin/vllm", "serve", MODEL,
     "--served-model-name", SERVED_NAME,
     "--dtype", "half",
     "--max-model-len", "16384",
     "--gpu-memory-utilization", "0.90",
     "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
     "--api-key", API_KEY,
     "--port", "8000"],
    stdout=open(f"{WORK}/vllm.log", "w"), stderr=subprocess.STDOUT,
)
print("vLLM started, pid", server.pid)

In [ ]:
import time, urllib.request

# Downloading the model (~5 GB) and loading it on the GPU takes a few minutes.
for waited in range(0, 900, 10):
    try:
        request = urllib.request.Request("http://localhost:8000/v1/models", headers={"Authorization": f"Bearer {API_KEY}"})
        print(urllib.request.urlopen(request, timeout=5).read().decode()[:300])
        print(f"ready after {waited}s")
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("vLLM stopped: read the end of vllm.log below")
        time.sleep(10)
else:
    raise RuntimeError("vLLM did not answer in 15 minutes: read vllm.log")

In [ ]:
!tail -n 15 {WORK}/vllm.log

## 5. Run the benchmark against the served model

The project's gateway routes `vllm:<name>` to `VLLM_BASE_URL`. Langfuse tracing is off; no Groq key is needed.

In [ ]:
env = {**os.environ,
       "VLLM_BASE_URL": "http://localhost:8000/v1",
       "VLLM_API_KEY": API_KEY,
       "REVIEW_LLM_TIMEOUT_S": "300",
       "LANGFUSE_TRACING_ENABLED": "false"}
command = [f"{TMP}/app-env/bin/python", "-m", "eval.llm_benchmark",
           "--models", f"vllm:{SERVED_NAME}", "--runs", str(RUNS)]
if ONLY:
    command += ["--only", ONLY]
subprocess.run(command, cwd=f"{TMP}/reviewer", env=env, check=True)

## 6. Keep the results

Copies the result files and the server log to `/kaggle/working`, so they appear under **Output** (right panel) to download. Bring the `.json` back to `eval/results/` on your laptop.

In [ ]:
!cp {TMP}/reviewer/eval/results/llm_benchmark_*.json {TMP}/reviewer/eval/results/llm_benchmark_*.md {WORK}/
!grep -E "throughput|Avg prompt" {WORK}/vllm.log | tail -n 5
!ls -la {WORK}

## 7. Stop the server

Frees the GPU. Also stop the session (top right, ⏻) when done: the weekly GPU hours count while it runs.

In [ ]:
server.terminate()
server.wait(timeout=60)
print("vLLM stopped")